In [7]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
import pyttsx3

In [8]:
model = load_model("gesture_model_3.keras")
img_size = 128 
class_names = ['approve', 'call_me', 'disapprove', 'fist', 'one', 'loser', 'ok', 'peace', 'rock', 'stop']

In [9]:
def predict_frame(frame):
  img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  img = cv2.resize(img, (img_size, img_size))
  img = img / 255.0
  img = np.expand_dims(img, axis = 0)

  predictions = model.predict(img, verbose=0)
  confidence = np.max(predictions)
  class_index = np.argmax(predictions)
  gesture = class_names[class_index]

  return gesture, confidence

In [10]:
last_prediction = ""
stable_count = 0

In [11]:
cap = cv2.VideoCapture(0)

engine = pyttsx3.init()
engine.setProperty('rate', 150)  

last_spoken = ""  

while True:
  ret, frame = cap.read()
  if not ret:
     break
  frame = cv2.flip(frame, 1)

  h, w, _ = frame.shape

  # ROI
  box_size = 250
  margin = 20
  x1 = w - box_size - margin
  y1 = margin
  x2 = w - margin
  y2 = margin + box_size

  cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

  roi = frame[y1:y2, x1:x2]

  gesture, confidence = predict_frame(roi)

  if confidence > 0.6:
    if gesture == last_prediction:
        stable_count += 1
    else:
        stable_count = 1
        last_prediction = gesture
    
    if stable_count > 10:
        print("Confirmed: ", gesture)
        if gesture != last_spoken:          
          engine.say(gesture.replace('_', ' '))
          engine.runAndWait()
          last_spoken = gesture
        stable_count = 0

  if confidence > 0.6:
    cv2.putText(frame, f"{gesture} ({confidence:.2f})",
                (x1, y1+300),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,0), 2)
    
  cv2.imshow("Gesture Recognition", frame)
  if cv2.waitKey(1) & 0xFF == 27:
    break

cap.release()
cv2.destroyAllWindows()

Confirmed:  one
Confirmed:  one
Confirmed:  one
Confirmed:  stop
Confirmed:  loser
Confirmed:  loser
Confirmed:  loser
Confirmed:  loser
Confirmed:  rock
Confirmed:  rock
Confirmed:  rock
Confirmed:  ok
